# Semana 5 — Busca Local: Hill Climbing e Simulated Annealing
## INF0415 — Heurísticas e Modelagem Multiobjetivo | UFG — BIA

**Objetivo:** Implementar Hill Climbing e Simulated Annealing para funções contínuas e TSP.

**Avaliação:** TL supervisionada (10 pts).

---

## 1. Imports e Configuração

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import random, math, time
from typing import List, Tuple

np.random.seed(42)
random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ Configuração carregada!')

## 2. Função de Rastrigin (fornecida)

Função multimodal com muitos ótimos locais — ideal para testar busca local.

$f(x) = 10n + \sum_{i=1}^{n} [x_i^2 - 10\cos(2\pi x_i)]$

Mínimo global: f(0, 0) = 0. Usamos -f (maximização → minimização).

In [ ]:
def rastrigin(x):
    """Função de Rastrigin. Mínimo global em x=0 com f=0."""
    A = 10
    return A * len(x) + sum(xi**2 - A * np.cos(2 * np.pi * xi) for xi in x)

def plot_rastrigin_2d():
    x = np.linspace(-5.12, 5.12, 200)
    y = np.linspace(-5.12, 5.12, 200)
    X, Y = np.meshgrid(x, y)
    Z = 20 + X**2 - 10*np.cos(2*np.pi*X) + Y**2 - 10*np.cos(2*np.pi*Y)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.contourf(X, Y, Z, levels=30, cmap='viridis')
    ax1.set_title('Rastrigin 2D — Contorno')
    ax1.set_xlabel('x₁'); ax1.set_ylabel('x₂')
    ax1.colorbar = plt.colorbar(ax1.contourf(X, Y, Z, levels=30, cmap='viridis'), ax=ax1)
    
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
    ax2.set_title('Rastrigin 2D — Superfície')
    plt.tight_layout()
    plt.show()

plot_rastrigin_2d()

## 3. Hill Climbing para Rastrigin 🔧

### ✏️ IMPLEMENTEM AQUI

Minimizar a função de Rastrigin usando steepest-ascent (na verdade, steepest-descent).
Vizinhança: somar vetor aleatório ~N(0, σ) ao ponto atual.

In [ ]:
def hill_climbing_continuous(f, x0, sigma=0.5, n_neighbors=20, max_iter=500):
    """Hill Climbing para minimização de função contínua."""
    x_current = np.array(x0, dtype=float)
    f_current = f(x_current)
    history = [(x_current.copy(), f_current)]
    
    # TODO: Implementar o loop principal
    # Para cada iteração:
    #   1. Gerar n_neighbors vizinhos (x_current + N(0, sigma))
    #   2. Clip para [-5.12, 5.12]
    #   3. Encontrar o melhor vizinho
    #   4. Se melhor que atual, mover. Senão, parar.
    #   5. Registrar em history
    
    pass  # remover ao implementar
    
    return x_current, f_current, history

### Teste do Hill Climbing

In [ ]:
x0 = [4.0, 4.0]  # ponto inicial longe do ótimo
x_best, f_best, hist = hill_climbing_continuous(rastrigin, x0)
print(f'Ponto inicial: {x0}, f = {rastrigin(x0):.4f}')
print(f'HC encontrou:  {x_best}, f = {f_best:.4f}')
print(f'Ótimo global:  [0, 0], f = 0.0000')
print(f'Iterações: {len(hist)}')
print(f'\n{"✅ Próximo do ótimo!" if f_best < 1.0 else "⚠️ Preso em ótimo local (esperado para HC!)"}')

# Visualizar
if len(hist) > 1:
    xs = [h[0][0] for h in hist]
    ys = [h[0][1] for h in hist]
    fs = [h[1] for h in hist]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x = np.linspace(-5.12, 5.12, 200)
    y = np.linspace(-5.12, 5.12, 200)
    X, Y = np.meshgrid(x, y)
    Z = 20 + X**2 - 10*np.cos(2*np.pi*X) + Y**2 - 10*np.cos(2*np.pi*Y)
    ax1.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.7)
    ax1.plot(xs, ys, 'r.-', markersize=4, linewidth=1, label='Trajetória HC')
    ax1.plot(xs[0], ys[0], 'go', markersize=10, label='Início')
    ax1.plot(xs[-1], ys[-1], 'r*', markersize=15, label='Final')
    ax1.legend(); ax1.set_title('Trajetória do HC')
    ax2.plot(fs, 'b-'); ax2.set_xlabel('Iteração'); ax2.set_ylabel('f(x)')
    ax2.set_title('Convergência do HC'); plt.tight_layout(); plt.show()

## 4. Simulated Annealing para Rastrigin 🔧

### ✏️ IMPLEMENTEM AQUI

**Nova estrutura:** a cada temperatura T, exploramos **N vizinhos** antes de resfriar.
Isso implementa o conceito de *equilíbrio térmico* — mais amostras por nível de temperatura.

A função `get_n_neighbors_continuous` já está fornecida. Implemente apenas o SA.

- Aceitar pioras com `P = exp(-Δ/T)`
- Se o custo do vizinho **empatar** com o atual (`parar_no_plato`), contar o platô
- Parar de explorar vizinhos quando `parar_no_plato == plateau_limit`

In [ ]:
def get_n_neighbors_continuous(x, n, sigma=0.5, bounds=(-5.12, 5.12)):
    """Gera N vizinhos de x somando ruído Gaussiano N(0, sigma)."""
    neighbors = []
    for _ in range(n):
        viz = x + np.random.normal(0, sigma, size=len(x))
        viz = np.clip(viz, bounds[0], bounds[1])
        neighbors.append(viz)
    return neighbors

# Teste rápido
x_teste = np.array([2.0, 3.0])
vizinhos_teste = get_n_neighbors_continuous(x_teste, n=5)
print(f'Ponto base: {x_teste}')
for i, v in enumerate(vizinhos_teste):
    print(f'  Vizinho {i+1}: {np.round(v, 4)}  |  dist={np.linalg.norm(v - x_teste):.4f}')

In [ ]:
def simulated_annealing_continuous(f, x0, T0=100.0, alpha=0.95, sigma=0.5,
                                    n_neighbors=20, min_T=0.1, plateau_limit=20):
    """
    Simulated Annealing para minimização de função contínua.
    A cada temperatura T, explora n_neighbors vizinhos antes de resfriar.
    Critério de platô: para de explorar se encontrar plateau_limit empates seguidos.
    """
    x_current = np.array(x0, dtype=float)
    f_current = f(x_current)
    x_best, f_best = x_current.copy(), f_current
    T = T0
    history = [(x_current.copy(), f_current, T)]

    # TODO: loop externo — enquanto T > min_T:
    #   1. Obter vizinhos: get_n_neighbors_continuous(x_current, n_neighbors, sigma)
    #   2. Inicializar parar_no_plato = 0
    #   3. Loop interno pelos vizinhos:
    #       a. Se parar_no_plato == plateau_limit: break
    #       b. Calcular probabilidade = exp(-(f_viz - f_current) / T)
    #          Dica: use math.exp com max(T, 1e-10) para evitar divisão por zero
    #       c. Aceitar se f_viz <= f_current OU random.random() < probabilidade
    #          - Se aceitar E f_viz == f_current: parar_no_plato += 1
    #          - Se aceitar E f_viz != f_current: parar_no_plato = 0
    #          - Atualizar x_current e f_current
    #          - Atualizar x_best, f_best se f_current < f_best
    #       d. Registrar (x_current.copy(), f_current, T) em history
    #   4. T = T * alpha

    pass  # remover ao implementar

    return x_best, f_best, history

### Teste e Comparação HC vs SA

In [ ]:
x0 = [4.0, 4.0]

# SA
x_sa, f_sa, hist_sa = simulated_annealing_continuous(rastrigin, x0, T0=100, alpha=0.995, max_iter=10000)
print(f'SA encontrou: x={x_sa}, f={f_sa:.4f}')
print(f'HC encontrou: f={f_best:.4f}' if 'f_best' in dir() else '')
print(f'Ótimo global: f=0.0000')

# Plotar convergência
if len(hist_sa) > 1:
    fs_sa = [h[1] for h in hist_sa]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(fs_sa, 'r-', alpha=0.5, linewidth=0.5, label='SA')
    if 'hist' in dir() and len(hist) > 1:
        fs_hc = [h[1] for h in hist]
        ax.plot(fs_hc, 'b-', linewidth=2, label='HC')
    ax.set_xlabel('Iteração'); ax.set_ylabel('f(x)')
    ax.set_title('Convergência: HC vs SA na Rastrigin')
    ax.legend(); plt.tight_layout(); plt.show()

## 5. TSP — Problema do Caixeiro Viajante (fornecido)

In [ ]:
class TSP:
    def __init__(self, n_cities=20, seed=42):
        np.random.seed(seed)
        self.n = n_cities
        self.coords = np.random.rand(n_cities, 2) * 100
        self.dist_matrix = np.zeros((n_cities, n_cities))
        for i in range(n_cities):
            for j in range(n_cities):
                self.dist_matrix[i][j] = np.linalg.norm(self.coords[i] - self.coords[j])
    
    def tour_cost(self, tour):
        cost = sum(self.dist_matrix[tour[i]][tour[i+1]] for i in range(len(tour)-1))
        cost += self.dist_matrix[tour[-1]][tour[0]]  # volta ao início
        return cost
    
    def random_tour(self):
        tour = list(range(self.n))
        random.shuffle(tour)
        return tour
    
    def plot_tour(self, tour, title='Tour', ax=None):
        if ax is None: fig, ax = plt.subplots(figsize=(8, 6))
        coords = self.coords
        for i in range(len(tour)):
            j = (i + 1) % len(tour)
            ax.plot([coords[tour[i],0], coords[tour[j],0]],
                    [coords[tour[i],1], coords[tour[j],1]], 'b-', linewidth=1)
        ax.scatter(coords[:,0], coords[:,1], c='red', s=50, zorder=5)
        for i, (x, y) in enumerate(coords):
            ax.annotate(str(i), (x, y), textcoords='offset points', xytext=(5,5), fontsize=8)
        ax.set_title(f'{title}\nCusto: {self.tour_cost(tour):.2f}')
        ax.set_aspect('equal')

tsp = TSP(n_cities=20)
print(f'TSP com {tsp.n} cidades')
random_tour = tsp.random_tour()
tsp.plot_tour(random_tour, 'Tour Aleatório')
plt.show()

## 6. Vizinhança 2-opt para TSP 🔧

### ✏️ IMPLEMENTEM AQUI

2-opt: escolher duas posições i < j no tour, inverter o segmento entre elas.

In [ ]:
def two_opt_swap(tour, i, j):
    """Aplica 2-opt: inverte o segmento tour[i:j+1]."""
    # TODO: retornar novo tour com segmento [i..j] invertido
    # Dica: tour[:i] + tour[i:j+1][::-1] + tour[j+1:]
    pass  # remover

def get_2opt_neighbors(tour):
    """Gera todos os vizinhos 2-opt do tour."""
    neighbors = []
    n = len(tour)
    # TODO: para cada par (i, j) com 1 <= i < j < n:
    #   gerar two_opt_swap(tour, i, j) e adicionar a neighbors
    pass  # remover
    return neighbors

## 7. Hill Climbing para TSP 🔧

### ✏️ IMPLEMENTEM AQUI

In [ ]:
def hill_climbing_tsp(tsp_instance, max_iter=1000):
    tour = tsp_instance.random_tour()
    cost = tsp_instance.tour_cost(tour)
    history = [cost]
    
    # TODO: loop principal
    # 1. Gerar todos os vizinhos 2-opt
    # 2. Encontrar o melhor (menor custo)
    # 3. Se melhor que atual, mover. Senão, parar.
    # 4. Registrar custo em history
    
    pass  # remover
    
    return tour, cost, history

## 8. Simulated Annealing para TSP 🔧

### ✏️ IMPLEMENTEM AQUI

Mesma estrutura do SA contínuo: **N vizinhos por temperatura**.
A função `get_n_neighbors_tsp` já está fornecida — ela gera N vizinhos 2-opt aleatórios.

In [ ]:
def get_n_neighbors_tsp(tour, n):
    """Gera N vizinhos do tour por 2-opt swap aleatório."""
    neighbors = []
    size = len(tour)
    for _ in range(n):
        i, j = sorted(random.sample(range(1, size), 2))
        neighbors.append(two_opt_swap(tour, i, j))
    return neighbors

# Teste rápido
tour_teste = tsp.random_tour()
viz_teste = get_n_neighbors_tsp(tour_teste, n=5)
print(f'Tour base:      custo = {tsp.tour_cost(tour_teste):.2f}')
for i, v in enumerate(viz_teste):
    print(f'  Vizinho {i+1}:  custo = {tsp.tour_cost(v):.2f}')

In [ ]:
def sa_tsp(tsp_instance, T0=1000.0, alpha=0.995, n_neighbors=30,
           min_T=0.1, plateau_limit=20):
    """
    Simulated Annealing para TSP.
    A cada temperatura T, explora n_neighbors vizinhos 2-opt antes de resfriar.
    Critério de platô: interrompe a exploração após plateau_limit empates consecutivos.
    """
    tour = tsp_instance.random_tour()
    cost = tsp_instance.tour_cost(tour)
    best_tour, best_cost = tour[:], cost
    T = T0
    history = [cost]

    # TODO: loop externo — enquanto T > min_T:
    #   1. Obter vizinhos: get_n_neighbors_tsp(tour, n_neighbors)
    #   2. Inicializar parar_no_plato = 0
    #   3. Loop interno pelos vizinhos:
    #       a. Se parar_no_plato == plateau_limit: break
    #       b. nc = tsp_instance.tour_cost(vizinho)
    #       c. delta = nc - cost
    #       d. probabilidade = exp(-delta / T)  (use max(T, 1e-10))
    #       e. Aceitar se nc <= cost OU random.random() < probabilidade
    #          - Se aceitar E nc == cost: parar_no_plato += 1
    #          - Se aceitar E nc != cost: parar_no_plato = 0
    #          - Atualizar tour e cost
    #          - Atualizar best_tour, best_cost se cost < best_cost
    #       f. Registrar cost em history
    #   4. T = T * alpha

    pass  # remover ao implementar

    return best_tour, best_cost, history

## 9. Experimento Comparativo 🧪

Execute esta célula para comparar HC vs SA no TSP.

In [ ]:
# Rodar HC e SA
t0 = time.time()
tour_hc, cost_hc, hist_hc = hill_climbing_tsp(tsp)
time_hc = time.time() - t0

t0 = time.time()
tour_sa, cost_sa, hist_sa = sa_tsp(tsp, T0=1000, alpha=0.995, max_iter=10000)
time_sa = time.time() - t0

print(f'HC: custo={cost_hc:.2f}, iter={len(hist_hc)}, tempo={time_hc:.3f}s')
print(f'SA: custo={cost_sa:.2f}, iter={len(hist_sa)}, tempo={time_sa:.3f}s')
print(f'Melhoria SA vs HC: {(cost_hc-cost_sa)/cost_hc*100:.1f}%')

# Visualizar tours
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tsp.plot_tour(tour_hc, 'Hill Climbing', ax=axes[0])
tsp.plot_tour(tour_sa, 'Simulated Annealing', ax=axes[1])
plt.tight_layout(); plt.show()

# Convergência
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_hc, 'b-', linewidth=2, label=f'HC (final: {cost_hc:.1f})')
ax.plot(hist_sa, 'r-', alpha=0.6, linewidth=0.5, label=f'SA (final: {cost_sa:.1f})')
ax.set_xlabel('Iteração'); ax.set_ylabel('Custo do Tour')
ax.set_title('Convergência: HC vs SA no TSP')
ax.legend(); plt.tight_layout(); plt.show()

## 10. Análise Estatística (10 execuções)

In [ ]:
n_runs = 10
costs_hc, costs_sa = [], []
for run in range(n_runs):
    _, c_hc, _ = hill_climbing_tsp(tsp)
    _, c_sa, _ = sa_tsp(tsp, T0=1000, alpha=0.995, max_iter=10000)
    costs_hc.append(c_hc)
    costs_sa.append(c_sa)

print(f'HC:  média={np.mean(costs_hc):.2f} ± {np.std(costs_hc):.2f}  '
      f'melhor={min(costs_hc):.2f}  pior={max(costs_hc):.2f}')
print(f'SA:  média={np.mean(costs_sa):.2f} ± {np.std(costs_sa):.2f}  '
      f'melhor={min(costs_sa):.2f}  pior={max(costs_sa):.2f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([costs_hc, costs_sa], labels=['Hill Climbing', 'Sim. Annealing'])
ax.set_ylabel('Custo do Tour'); ax.set_title('Distribuição de custos (10 execuções)')
plt.tight_layout(); plt.show()

## 11. Perguntas ✍️

### P1. O HC ficou preso em ótimos locais? Como você percebeu isso?
*Resposta:*

### P2. O SA encontrou soluções melhores que o HC? Em todas as execuções?
*Resposta:*

### P3. Qual o efeito de α (taxa de resfriamento) na qualidade? Teste α=0.9, 0.99, 0.999.
*Resposta:*

### P4. Como a paisagem de Rastrigin se compara com o TSP em termos de dificuldade para HC?
*Resposta:*

### P5. Se você pudesse combinar HC e SA, como faria? (Tease para hibridização na Sem. 7)
*Resposta:*

### P6. Qual a conexão entre a função de avaliação do Minimax (Sem. 4) e a fitness do HC?
*Resposta:*

---
## 📚 Para a Semana 6

**Busca Tabu:** memória de curto/longo prazo, intensificação, diversificação.

Usaremos o MESMO código TSP — apenas mudaremos a estratégia de aceitação.

**Leitura:** Goldbarg Cap. 5 OU Luke, Essentials of Metaheuristics Cap. 4.

---
*INF0415 — UFG 2026/2*